<a href="https://colab.research.google.com/github/mundundan-star/online-retail-customer-analytics/blob/main/7.0%20Retention_Priority_Matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from sqlalchemy import inspect, text,create_engine
from google.colab import drive
from google.colab import userdata

drive.mount('/content/drive')

clustering_model = joblib.load('/content/drive/MyDrive/clustering_model.joblib')
churn_model = joblib.load('/content/drive/MyDrive/churn_model.joblib')
frequency_model = joblib.load('/content/drive/MyDrive/frequency_regression_model.joblib')
aov_model = joblib.load('/content/drive/MyDrive/aov_regression_model.joblib')

engine = create_engine(userdata.get("NEON_DATABASE_URL"))

metrics_query = """
SELECT "CustomerID",
    SUM("Revenue") AS "TotalRevenue",
    AVG("Revenue") AS "AOV",
    MAX("InvoiceDate") - MIN("InvoiceDate") AS "ObservedLifeSpan",
    (SELECT MAX("InvoiceDate")
    FROM v_clean_sales_analytics) - MAX("InvoiceDate") AS "Recency",
    (SELECT MAX("InvoiceDate")
    FROM v_clean_sales_analytics) - MIN("InvoiceDate") AS "Tenure",
     COUNT(DISTINCT "StockCode") AS "ProductDiversity",
    COUNT(DISTINCT "InvoiceDate") AS "Frequency",
    TO_CHAR(MIN("InvoiceDate"), 'YYYY-MM') AS "CohortMonth"

FROM v_clean_sales_analytics
WHERE "InvoiceDate" > '2011-08-31'
GROUP BY "CustomerID"
"""

metrics_df = pd.read_sql(text(metrics_query), con = engine)


rfm_query = """WITH metrics AS
(SELECT "CustomerID",
    (SELECT MAX("InvoiceDate")
    FROM v_clean_sales_analytics) - MAX("InvoiceDate") AS "Recency",
    COUNT(DISTINCT "InvoiceDate") AS "Frequency",
    SUM("Revenue") AS "Monetary"
FROM v_clean_sales_analytics
WHERE "InvoiceDate" > '2011-08-31'
GROUP BY "CustomerID"),

rfm_scores AS
(SELECT "CustomerID",
    NTILE(5) OVER(ORDER BY "Recency" DESC) AS "R",
    NTILE(5) OVER(ORDER BY "Frequency" ASC) AS "F",
    NTILE(5) OVER(ORDER BY "Monetary" ASC) AS "M"
FROM metrics)

SELECT *,
    "R" + "F" + "M" AS "RfmScore"
FROM rfm_scores
"""

rfm_df = pd.read_sql(text(rfm_query), con = engine)

#Predicting churn labels
X = metrics_df.drop(['CustomerID', 'AOV', 'CohortMonth'], axis =1)
X = pd.get_dummies(X, drop_first = True).values

metrics_df['KSegment'] = clustering_model.predict(X)

#Predicting churn labels
X = metrics_df.drop(['CustomerID', 'AOV', 'CohortMonth'], axis =1)
X = pd.get_dummies(X, drop_first = True).values

metrics_df['ChurnProbalilty'] = churn_model.predict_proba(X)[:,1]
metrics_df['ChurnLabel'] = churn_model.predict(X)



metrics_df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


ValueError: X has 7 features, but StandardScaler is expecting 17 features as input.

In [3]:
metrics_df['KSegment'].value_counts()

,count
KSegment,
1,1639
3,1044
2,291
